# Session 10 (Sessions review)
## Question 3 of 4 — Optional: Revisit a Previous Session Question (I chose session8)

### Extension: Verifying Backpropagation and Gradient Descent by Hand

For this extension, I revisit the gradient descent example from Session 8. My goal is to go beyond running the training loop; I want to verify the first update by hand and explain why the parameters move in the direction shown by the code.

I use one input vector,

$$
x = [-1,-2],
$$

with initial parameters

$$
w = [2,-3], \qquad b=-3,
$$

and target

$$
y=0.
$$

The model first computes

$$
z=w^Tx+b,
$$

and then applies the sigmoid function,

$$
f=\sigma(z)=\frac{1}{1+e^{-z}}.
$$

I use the squared-error loss

$$
L=\frac{1}{2}(f-y)^2.
$$

This small example lets me connect the forward pass, chain rule, backpropagation, and gradient-descent update directly.

In [1]:
import numpy as np

# Defining the input and target
x = np.array([-1.0, -2.0])
target = 0.0

# Initializing the parameters
w = np.array([2.0, -3.0])
b = -3.0

# Defining the learning rate
eta = 0.1

# Defining the sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Computing the initial forward pass
z = w.T @ x + b
f = sigmoid(z)
loss = 0.5 * (f - target)**2

print(f"z = {z:.4f}")
print(f"sigmoid output = {f:.4f}")
print(f"loss = {loss:.4f}")

z = 1.0000
sigmoid output = 0.7311
loss = 0.2672


## Hand Calculation of the First Gradient

The forward pass gives

$$
z=(2)(-1)+(-3)(-2)-3=1.
$$

Therefore,

$$
f=\sigma(1)\approx0.7311.
$$

Because the target is 0,

$$
L=\frac{1}{2}(0.7311-0)^2\approx0.2672.
$$

I now work backward using the chain rule.

First,

$$
\frac{\partial L}{\partial f}=f-y.
$$

So,

$$
\frac{\partial L}{\partial f}=0.7311.
$$

The derivative of the sigmoid is

$$
\frac{\partial f}{\partial z}=f(1-f).
$$

Therefore,

$$
\frac{\partial f}{\partial z}
=(0.7311)(1-0.7311)
\approx0.1966.
$$

Combining these derivatives gives

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial f}
\frac{\partial f}{\partial z}
\approx
(0.7311)(0.1966)
\approx0.1437.
$$

Since

$$
z=w^Tx+b,
$$

the weight gradient is

$$
\nabla_wL
=
\frac{\partial L}{\partial z}x.
$$

Thus,

$$
\nabla_wL
=
0.1437[-1,-2]
\approx
[-0.1437,-0.2875].
$$

For the bias,

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial z}
\approx0.1437.
$$

These values describe how sensitive the current loss is to small changes in each parameter.

In [2]:
# Computing the derivatives from the forward pass
d_f = f - target
d_z = d_f * f * (1 - f)

# Computing the weight and bias gradients
d_w = d_z * x
d_b = d_z

# Displaying the gradients
print(f"dL/df = {d_f:.4f}")
print(f"df/dz = {f * (1 - f):.4f}")
print(f"dL/dz = {d_z:.4f}")
print(f"dL/dw = {d_w}")
print(f"dL/db = {d_b:.4f}")

dL/df = 0.7311
df/dz = 0.1966
dL/dz = 0.1437
dL/dw = [-0.14373484 -0.28746968]
dL/db = 0.1437


## Verifying the First Gradient-Descent Update

Gradient descent updates each parameter using

$$
\theta_{\text{new}}
=
\theta_{\text{old}}
-
\eta\nabla_{\theta}L.
$$

Using

$$
\eta=0.1,
$$

the first weight becomes

$$
w_{1,\text{new}}
=
2-0.1(-0.1437)
\approx2.0144.
$$

The second weight becomes

$$
w_{2,\text{new}}
=
-3-0.1(-0.2875)
\approx-2.9713.
$$

The bias becomes

$$
b_{\text{new}}
=
-3-0.1(0.1437)
\approx-3.0144.
$$

The important point is that the gradients should not equal the updated weights. The gradients tell us the direction and size of the local change in the loss; gradient descent then uses those gradients to calculate new parameter values.

In [3]:
# Computing the first parameter update
w_new = w - eta * d_w
b_new = b - eta * d_b

# Displaying the updated parameters
print(f"old weights = {w}")
print(f"weight gradient = {d_w}")
print(f"new weights = {w_new}")

print(f"\nold bias = {b:.4f}")
print(f"bias gradient = {d_b:.4f}")
print(f"new bias = {b_new:.4f}")

old weights = [ 2. -3.]
weight gradient = [-0.14373484 -0.28746968]
new weights = [ 2.01437348 -2.97125303]

old bias = -3.0000
bias gradient = 0.1437
new bias = -3.0144


## Why Does the Update Move in This Direction?

The direction of the update also makes mathematical sense.

The target is

$$
y=0,
$$

but the current prediction is

$$
f\approx0.7311.
$$

Therefore, I want the sigmoid output to decrease. Since the sigmoid is an increasing function, decreasing its output requires decreasing its input,

$$
z=w^Tx+b.
$$

Both input values are negative:

$$
x_1=-1,\qquad x_2=-2.
$$

The two weight gradients are also negative, so gradient descent increases both weights. Multiplying larger positive weights by negative inputs makes

$$
w^Tx
$$

more negative.

The bias gradient is positive, so gradient descent decreases the bias. This also makes $z$ smaller.

Therefore, both changes push

$$
z
$$

downward, which pushes the sigmoid prediction closer to the target of 0.

This gives me an intuitive check on the calculus; the signs of the gradients agree with what I expect the model to do.

In [4]:
# Comparing the model before and after one update
z_before = w.T @ x + b
f_before = sigmoid(z_before)

z_after = w_new.T @ x + b_new
f_after = sigmoid(z_after)

print(f"z before update = {z_before:.4f}")
print(f"output before update = {f_before:.4f}")

print(f"\nz after update = {z_after:.4f}")
print(f"output after update = {f_after:.4f}")

z before update = 1.0000
output before update = 0.7311

z after update = 0.9138
output after update = 0.7138


## Running Gradient Descent for 1,000 Iterations

After checking the first update manually, I run the same calculation repeatedly.

At every iteration, the model:

1. computes the current value of $z$;
2. applies the sigmoid;
3. calculates the loss;
4. computes the derivatives using the chain rule;
5. updates the weights and bias.

The update rules are

$$
w \leftarrow w-\eta\nabla_wL
$$

and

$$
b \leftarrow b-\eta\frac{\partial L}{\partial b}.
$$

I print the progress every 100 iterations so I can check whether the output and loss are moving in the expected direction.

In [5]:
# Reinitializing the parameters before training
w = np.array([2.0, -3.0])
b = -3.0

# Setting the training values
eta = 0.1
n_iter = 1000

# Running gradient descent
for i in range(n_iter):
    # Computing the forward pass
    z = w.T @ x + b
    f = sigmoid(z)

    # Computing the squared-error loss
    loss = 0.5 * (f - target)**2

    # Computing the chain-rule gradients
    d_f = f - target
    d_z = d_f * f * (1 - f)
    d_w = d_z * x
    d_b = d_z

    # Updating the parameters
    w -= eta * d_w
    b -= eta * d_b

    # Displaying training progress
    if i % 100 == 0:
        print(f"Iteration {i}: output={f:.4f}, loss={loss:.6f}, w={w}, b={b:.4f}")

Iteration 0: output=0.7311, loss=0.267223, w=[ 2.01437348 -2.97125303], b=-3.0144
Iteration 100: output=0.1181, loss=0.006972, w=[ 2.50301232 -1.99397536], b=-3.5030
Iteration 200: output=0.0765, loss=0.002927, w=[ 2.58232644 -1.83534713], b=-3.5823
Iteration 300: output=0.0603, loss=0.001815, w=[ 2.62484135 -1.7503173 ], b=-3.6248
Iteration 400: output=0.0511, loss=0.001307, w=[ 2.65376944 -1.69246111], b=-3.6538
Iteration 500: output=0.0451, loss=0.001017, w=[ 2.67563893 -1.64872213], b=-3.6756
Iteration 600: output=0.0408, loss=0.000831, w=[ 2.69319331 -1.61361339], b=-3.6932
Iteration 700: output=0.0375, loss=0.000702, w=[ 2.70784108 -1.58431783], b=-3.7078
Iteration 800: output=0.0348, loss=0.000607, w=[ 2.72040005 -1.55919989], b=-3.7204
Iteration 900: output=0.0327, loss=0.000534, w=[ 2.73138659 -1.53722682], b=-3.7314


## Checking the Final Model

After training, I recompute the prediction using the final parameter values.

The prediction should decrease from approximately

$$
0.7311
$$

at the beginning to approximately

$$
0.0309.
$$

The loss should also become much smaller.

Because a sigmoid approaches 0 asymptotically, I do not expect the prediction to become exactly zero. What matters is that repeated gradient-descent updates consistently move the model toward the target.

In [6]:
# Recomputing the output with the final parameters
final_z = w.T @ x + b
final_output = sigmoid(final_z)
final_loss = 0.5 * (final_output - target)**2

# Displaying the final result
print(f"Final output = {final_output:.6f}")
print(f"Final loss = {final_loss:.6f}")
print(f"Final weights = {w}")
print(f"Final bias = {b:.6f}")

Final output = 0.030879
Final loss = 0.000477
Final weights = [ 2.74105493 -1.51789014]
Final bias = -3.741055


## Interpretation and Justification

This extension helped me connect the calculus directly to the behavior of the training code.

First, I calculated the derivatives by hand and obtained

$$
\nabla_wL\approx[-0.1437,-0.2875]
$$

and

$$
\frac{\partial L}{\partial b}\approx0.1437.
$$

Using those gradients with a learning rate of $0.1$ produced the same first parameter update as the code. This confirms that the implementation is following the chain rule and gradient-descent equations correctly.

I also checked the direction of the update instead of only comparing numerical values. Since the target is 0, the model needs a smaller sigmoid output. The negative inputs mean that increasing the weights makes $w^Tx$ more negative; decreasing the bias pushes $z$ downward as well. Both changes therefore move the prediction in the correct direction.

Across 1,000 iterations, the output falls from about $0.7311$ to $0.0309$, while the loss becomes very small. This shows that the repeated local gradient calculations consistently reduce the error.

I chose this extension because it lets me treat the training loop as something I can explain mathematically, rather than as a black box. The handwritten derivatives predict the first update, the signs of the gradients explain the direction of movement, and the full training loop shows the same reasoning operating repeatedly.